# Per-Step Hyperparameter Control & Caching Test Notebook

This notebook tests the new per-step configuration system including:
- `StepConfig` and `AgentConfig` dataclasses
- Per-step best-of-n execution
- Result caching and selective re-execution
- Evaluator factory functions
- Backward compatibility

## 1. Setup and Imports

**Purpose:** Import all required modules from the `Agent` package, including the three new classes introduced by the per-step configuration refactor — `AgentConfig`, `StepConfig`, and `RunCache`. The `sys.path` insertion ensures Python loads the local development version of the module rather than any installed package version. A successful import confirms the module structure is intact and all new files are accessible.

In [1]:
import os
import sys
import tempfile
import shutil

# Ensure we're using the local Agent module
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))

# Import the new classes
from Agent import SalesDataAgent, State, AgentConfig, StepConfig, RunCache
from Agent.utils import make_csv_evaluator, make_text_evaluator

print("All imports successful!")

CodeCarbon is available
<module 'langgraph.version' from '/home/oss/Desktop/DataAgent/venv/lib64/python3.12/site-packages/langgraph/version.py'>
All imports successful!


**Purpose:** Create a `StepConfig` instance and inspect its fields. `StepConfig` controls how a single agent step executes: `n` sets how many best-of-n candidates to generate, `temp_min`/`temp_max` define the temperature range spread across those candidates, and `max_tokens` caps response length. The key method `get_temperatures()` returns a linearly spaced list of temperature values — one per candidate — so each best-of-n run uses a slightly different temperature.

**Purpose:** Verify that `StepConfig` can be serialized to a plain Python dict (stripping non-serializable callables like `eval_fn`) and correctly deserialized back from that dict. This round-trip is essential for the cache system, which stores all configurations as JSON alongside the step results so past runs can be fully reconstructed and compared.

## 2. Test StepConfig Creation and Serialization

In [2]:
# Test basic StepConfig creation
config = StepConfig(
    n=5,
    temp_min=0.1,
    temp_max=0.9,
    max_tokens=2500,
    step_name="test_step"
)

print(f"StepConfig created:")
print(f"  n = {config.n}")
print(f"  temp_min = {config.temp_min}")
print(f"  temp_max = {config.temp_max}")
print(f"  max_tokens = {config.max_tokens}")
print(f"  use_cache = {config.use_cache}")
print(f"  cache_mode = {config.cache_mode}")
print(f"  enabled = {config.enabled}")
print(f"\nTemperatures for best-of-{config.n}: {config.get_temperatures()}")

StepConfig created:
  n = 5
  temp_min = 0.1
  temp_max = 0.9
  max_tokens = 2500
  use_cache = True
  cache_mode = auto
  enabled = True

Temperatures for best-of-5: [0.1, 0.30000000000000004, 0.5, 0.7000000000000001, 0.9]


**Purpose:** Show that individual step configurations inside an `AgentConfig` can be freely modified after creation. Because each step config is a separate `StepConfig` object, changing `analyzing_data.n` does not affect any other step. The full config can also be serialized to a dict, which is what the cache writes to disk alongside the run results.

**Purpose:** Verify that `AgentConfig` defaults to `gpt-4o-mini` as the model and `openai` as the provider. `AgentConfig` is the top-level configuration container: it holds one `StepConfig` per agent step (`decide_tool`, `lookup_sales_data`, `analyzing_data`, `create_visualization`) plus global LLM settings. The assertions confirm the defaults are correctly set after the refactor.

In [3]:
# Test serialization
config_dict = config.to_dict()
print("Serialized config:")
for k, v in config_dict.items():
    print(f"  {k}: {v}")

# Test deserialization
restored_config = StepConfig.from_dict(config_dict)
print(f"\nDeserialized: n={restored_config.n}, temps={restored_config.get_temperatures()}")

Serialized config:
  n: 5
  temp_min: 0.1
  temp_max: 0.9
  max_tokens: 2500
  top_p: 1.0
  use_cache: True
  cache_mode: auto
  enabled: True
  step_name: test_step

Deserialized: n=5, temps=[0.1, 0.30000000000000004, 0.5, 0.7000000000000001, 0.9]


**Purpose:** Save a simulated agent run to the cache, including multiple candidate results per step (as produced by best-of-n sampling). The cache stores ALL N candidates — not just the selected best — so that a future run can re-evaluate them with a different scoring function without calling the LLM again. The `_temperature` field embedded in each result records which temperature generated it.

**Purpose:** Load and inspect the cached step results for specific steps. This verifies that `load_step_results()` correctly deserializes the JSON file for a given step and returns the full list of N candidates. The temperature and content of each candidate are checked to confirm nothing was lost or corrupted during the save/load cycle.

**Purpose:** Test the prompt similarity search used by the cache's auto-discovery feature. Given a new query, `find_similar_runs()` uses Jaccard similarity on word tokens to identify past runs with overlapping vocabulary. This allows the agent to automatically suggest reusable cached results when a user runs a similar (but not identical) query, without needing to manually specify a run ID.

**Purpose:** Remove the temporary cache directory created for this test section. Since `RunCache` writes real files to disk, cleanup is necessary to avoid accumulating test artifacts. In production, the cache directory persists intentionally across runs — only test code should clean it up.

## 3. Test AgentConfig Creation

**Purpose:** Initialize a `RunCache` pointing at a temporary directory. `RunCache` manages a persistent directory structure: an `index.json` for fast prompt lookups and per-run subdirectories each containing step-level JSON files. Using a temp directory here keeps the test self-contained and avoids polluting the project cache.

In [4]:
# Test AgentConfig with defaults
agent_config = AgentConfig()

print("Default AgentConfig:")
print(f"  model = {agent_config.model}")
print(f"  provider = {agent_config.provider}")
print(f"\nStep configurations:")
for step_name in ['decide_tool', 'lookup_sales_data', 'analyzing_data', 'create_visualization']:
    step_config = agent_config.get_step_config(step_name)
    print(f"  {step_name}: n={step_config.n}, temp_range=[{step_config.temp_min}, {step_config.temp_max}], use_cache={step_config.use_cache}")

assert agent_config.model == "gpt-4o-mini", f"Expected gpt-4o-mini, got {agent_config.model}"
assert agent_config.provider == "openai", f"Expected openai, got {agent_config.provider}"
print("\nDefault model and provider verified!")

Default AgentConfig:
  model = gpt-4o-mini
  provider = openai

Step configurations:
  decide_tool: n=1, temp_range=[0.1, 0.1], use_cache=False
  lookup_sales_data: n=1, temp_range=[0.1, 0.3], use_cache=True
  analyzing_data: n=1, temp_range=[0.1, 0.7], use_cache=True
  create_visualization: n=1, temp_range=[0.1, 0.5], use_cache=True

Default model and provider verified!


In [5]:
# Test modifying step configs
agent_config.analyzing_data.n = 3
agent_config.analyzing_data.temp_max = 0.7
agent_config.lookup_sales_data.n = 2

print("Modified AgentConfig:")
print(f"  analyzing_data: n={agent_config.analyzing_data.n}, temp_max={agent_config.analyzing_data.temp_max}")
print(f"  lookup_sales_data: n={agent_config.lookup_sales_data.n}")

# Test serialization
full_config = agent_config.to_dict()
print(f"\nSerialized keys: {list(full_config.keys())}")

Modified AgentConfig:
  analyzing_data: n=3, temp_max=0.7
  lookup_sales_data: n=2

Serialized keys: ['decide_tool', 'lookup_sales_data', 'analyzing_data', 'create_visualization', 'model', 'provider', 'ollama_url']


**Purpose:** Instantiate `SalesDataAgent` with the custom `AgentConfig` and a dedicated cache directory. The agent stores the config internally and sets up the cache. No API calls are made during initialization — the agent only contacts OpenAI when `run()` is called. Requires `OPENAI_API_KEY` to be set in the environment; otherwise the agent will fail when `run()` is called (not during `__init__`).

## 4. Test RunCache

In [6]:
# Create a temporary cache directory for testing
test_cache_dir = tempfile.mkdtemp(prefix="agent_cache_test_")
cache = RunCache(test_cache_dir)

print(f"Cache directory: {cache.cache_dir}")
print(f"Initial stats: {cache.get_cache_stats()}")

Cache directory: /tmp/agent_cache_test_cq4yieiq
Initial stats: {'total_runs': 0, 'total_size_bytes': 0, 'total_size_mb': 0.0, 'cache_dir': '/tmp/agent_cache_test_cq4yieiq'}


In [7]:
# Test saving a run
test_run_id = "test123"
test_prompt = "Show total sales in November 2021"

# Simulate step results (all N results from best-of-n)
step_results = {
    "lookup_sales_data": [
        {"sql_query": "SELECT * FROM sales WHERE month='Nov'", "data": "row1\nrow2", "_temperature": 0.1},
        {"sql_query": "SELECT SUM(amount) FROM sales WHERE month='November'", "data": "total: 1000", "_temperature": 0.3},
    ],
    "analyzing_data": [
        {"answer": ["Sales were $1000 in November."], "_temperature": 0.1},
        {"answer": ["November 2021 saw total sales of $1000."], "_temperature": 0.4},
        {"answer": ["The total revenue for Nov 2021 was approximately $1000."], "_temperature": 0.7},
    ]
}

final_result = {
    "answer": ["Sales were $1000 in November."],
    "data": "total: 1000"
}

cache.save_run(
    run_id=test_run_id,
    prompt=test_prompt,
    agent_config=agent_config.to_dict(),
    step_results=step_results,
    final_result=final_result
)

print(f"Run saved!")
print(f"Cache stats: {cache.get_cache_stats()}")

Run saved!
Cache stats: {'total_runs': 1, 'total_size_bytes': 2005, 'total_size_mb': 0.0, 'cache_dir': '/tmp/agent_cache_test_cq4yieiq'}


**Purpose:** Test `make_text_evaluator`, which is a factory function that returns a scoring function comparing the agent's text answer to a ground-truth reference using BLEU score. This is used as an `eval_fn` inside `StepConfig` to automatically score and rank best-of-n candidates. BLEU scores range from 0 (no match) to 1 (exact match), so the exact-match hypothesis should score highest.

**Purpose:** Run a full end-to-end agent query using `gpt-4o-mini` with per-step best-of-n enabled. `save_results=True` writes the run's step results to the cache under an auto-generated `run_id`. After the call completes, the test inspects `current_run_step_results` to confirm that each step stored its candidates — which is the data that the cache reuse test below will reload.

**Purpose:** Demonstrate selective cache reuse — the core experimental workflow the refactor enables. The first run executes all steps fresh and saves results. The second run reuses the cached SQL result from `lookup_sales_data` (`cache_mode="skip"`) but forces a fresh LLM call for `analyzing_data` (`cache_mode="force_fresh"`). This lets you re-experiment with analysis quality without paying the cost of re-running SQL generation against the database.

**Purpose:** Demonstrate per-run step overrides without permanently modifying the agent's configuration. `step_overrides` is applied as a temporary deep copy of the config before the run and automatically restored afterwards. This is useful for one-off experiments (e.g. "try n=3 just this once") without having to modify and then restore the agent config manually. The final assertion confirms the config reverts to `n=1` after the run.

In [8]:
# Test loading step results
loaded_lookup = cache.load_step_results(test_run_id, "lookup_sales_data")
loaded_analysis = cache.load_step_results(test_run_id, "analyzing_data")

print(f"Loaded lookup_sales_data results: {len(loaded_lookup)} items")
for i, result in enumerate(loaded_lookup):
    print(f"  Result {i+1}: T={result.get('_temperature')}, SQL: {result.get('sql_query', '')[:50]}...")

print(f"\nLoaded analyzing_data results: {len(loaded_analysis)} items")
for i, result in enumerate(loaded_analysis):
    print(f"  Result {i+1}: T={result.get('_temperature')}, Answer: {result.get('answer', [''])[0][:50]}...")

Loaded lookup_sales_data results: 2 items
  Result 1: T=0.1, SQL: SELECT * FROM sales WHERE month='Nov'...
  Result 2: T=0.3, SQL: SELECT SUM(amount) FROM sales WHERE month='Novembe...

Loaded analyzing_data results: 3 items
  Result 1: T=0.1, Answer: Sales were $1000 in November....
  Result 2: T=0.4, Answer: November 2021 saw total sales of $1000....
  Result 3: T=0.7, Answer: The total revenue for Nov 2021 was approximately $...


**Purpose:** Build a custom `AgentConfig` explicitly targeting `gpt-4o-mini` with OpenAI as the provider, then configure each step independently. The lookup step uses `n=1` since SQL generation is fairly deterministic; the analysis step uses `n=2` across a temperature range to explore different phrasings; the visualization step uses `n=1` for a single pass. This demonstrates the per-step granularity that is the core goal of the refactor.

In [9]:
# Test prompt similarity matching
similar_prompts = [
    "Show total sales in November 2021",  # Exact match
    "Display total sales for Nov 2021",   # Very similar
    "What were the sales in November?",   # Somewhat similar
    "Show revenue by product category",   # Different
]

print("Prompt similarity test:")
for prompt in similar_prompts:
    similar = cache.find_similar_runs(prompt, top_k=3)
    print(f"  '{prompt[:40]}...' -> {similar}")

Prompt similarity test:
  'Show total sales in November 2021...' -> ['test123']
  'Display total sales for Nov 2021...' -> ['test123']
  'What were the sales in November?...' -> []
  'Show revenue by product category...' -> []


In [10]:
# Clean up test cache
shutil.rmtree(test_cache_dir)
print(f"Test cache cleaned up")

Test cache cleaned up


**Purpose:** Confirm that the old initialization style (passing `model`, `temperature`, `max_tokens` as keyword arguments directly to `SalesDataAgent`) still works without any changes. The agent internally converts these arguments into an `AgentConfig`, so all existing code and notebooks using the old API continue to function unchanged. This is the backward-compatibility guarantee of the refactor.

## 5. Test Evaluator Factory Functions

**Purpose:** Check whether an `OPENAI_API_KEY` environment variable is available before attempting any real LLM calls. All subsequent integration test cells gate their execution on this flag (`if OPENAI_AVAILABLE`), so the notebook remains runnable in offline or CI environments where no API key is present — those cells will simply print a skip message.

In [11]:
# Test text evaluator factory
ground_truth = "The total sales in November 2021 were $1,234,567."

text_eval = make_text_evaluator(ground_truth, metric="bleu")

# Test with different hypotheses
test_results = [
    {"answer": ["The total sales in November 2021 were $1,234,567."]},  # Exact match
    {"answer": ["Total sales for Nov 2021: $1,234,567."]},  # Similar
    {"answer": ["Sales were good in November."]},  # Different
    {"answer": ["The weather was nice."]},  # Unrelated
]

print("Text evaluator test (BLEU scores):")
for result in test_results:
    score = text_eval(result, {})
    print(f"  '{result['answer'][0][:50]}...' -> {score:.4f}")

Text evaluator test (BLEU scores):
  'The total sales in November 2021 were $1,234,567....' -> 1.0000
  'Total sales for Nov 2021: $1,234,567....' -> 0.2873
  'Sales were good in November....' -> 0.1502
  'The weather was nice....' -> 0.0802


## 6. Test Agent Initialization with Config

In [12]:
# Create a custom AgentConfig
custom_config = AgentConfig(
    model="gpt-4o-mini",
    provider="openai"
)

# Configure per-step settings
custom_config.lookup_sales_data.n = 1  # Single run for lookup (deterministic SQL)
custom_config.analyzing_data.n = 2     # 2 analysis attempts
custom_config.analyzing_data.temp_min = 0.1
custom_config.analyzing_data.temp_max = 0.5
custom_config.create_visualization.n = 1  # Single visualization

print("Custom configuration:")
print(f"  model = {custom_config.model}")
print(f"  provider = {custom_config.provider}")
print(f"  lookup_sales_data: n={custom_config.lookup_sales_data.n}")
print(f"  analyzing_data: n={custom_config.analyzing_data.n}, temps={custom_config.analyzing_data.get_temperatures()}")
print(f"  create_visualization: n={custom_config.create_visualization.n}")

Custom configuration:
  model = gpt-4o-mini
  provider = openai
  lookup_sales_data: n=1
  analyzing_data: n=2, temps=[0.1, 0.5]
  create_visualization: n=1


In [13]:
# Initialize agent with custom config
# Note: Requires OPENAI_API_KEY environment variable to be set
test_cache_dir = tempfile.mkdtemp(prefix="agent_test_")

try:
    agent = SalesDataAgent(
        agent_config=custom_config,
        cache_dir=test_cache_dir
    )
    print("Agent initialized successfully!")
    print(f"  Agent model: {agent.model}")
    print(f"  Agent provider: {agent.provider}")
    print(f"  Agent config model: {agent.agent_config.model}")
    print(f"  Agent cache dir: {agent.cache.cache_dir}")
    print(f"  analyzing_data.n: {agent.agent_config.analyzing_data.n}")
except Exception as e:
    print(f"Agent initialization error: {e}")

Agent initialized successfully!
  Agent model: gpt-4o-mini
  Agent provider: openai
  Agent config model: gpt-4o-mini
  Agent cache dir: /tmp/agent_test_uxocuc95
  analyzing_data.n: 2


**Purpose:** Print a final confirmation that all cells in the notebook completed without raising uncaught exceptions. Because Python notebooks execute cells top-to-bottom and stop on errors, reaching this cell means every test section passed successfully.

## 7. Test Backward Compatibility

In [14]:
# Test that old initialization still works (with new defaults)
try:
    old_style_agent = SalesDataAgent(
        model="gpt-4o-mini",
        temperature=0.1,
        max_tokens=2000
    )
    print("Old-style initialization works!")
    print(f"  Model: {old_style_agent.model}")
    print(f"  Provider: {old_style_agent.provider}")
    print(f"  Has agent_config: {hasattr(old_style_agent, 'agent_config')}")
    print(f"  Has cache: {hasattr(old_style_agent, 'cache')}")
except Exception as e:
    print(f"Old-style initialization error: {e}")

Old-style initialization works!
  Model: gpt-4o-mini
  Provider: openai
  Has agent_config: True
  Has cache: True


## 8. Full Integration Test (Requires OpenAI API Key)

The following tests require a valid `OPENAI_API_KEY` environment variable and use `gpt-4o-mini`.

In [2]:
# Check if OpenAI API key is available
import os

OPENAI_AVAILABLE = bool(os.getenv("OPENAI_API_KEY"))
print(f"OpenAI API key available: {OPENAI_AVAILABLE}")
if not OPENAI_AVAILABLE:
    print("Set OPENAI_API_KEY environment variable to run integration tests")

OpenAI API key available: True


In [3]:
# Run integration test if OpenAI is available
if OPENAI_AVAILABLE:
    print("Running integration test with per-step best-of-n using gpt-4o-mini...")
    
    # Create config with multiple runs per step
    test_config = AgentConfig(model="gpt-4o-mini", provider="openai")
    test_config.lookup_sales_data.n = 1  # SQL usually deterministic
    test_config.analyzing_data.n = 2     # Test 2 analysis attempts
    test_config.analyzing_data.temp_min = 0.1
    test_config.analyzing_data.temp_max = 0.5
    test_config.create_visualization.n = 1
    
    integration_cache = tempfile.mkdtemp(prefix="integration_test_")
    
    agent = SalesDataAgent(
        agent_config=test_config,
        cache_dir=integration_cache
    )
    
    # Run with caching enabled
    result = agent.run(
        "What were the total sales in November 2021?",
        no_vis=True,  # Skip visualization for faster test
        save_results=True
    )
    
    print(f"\nRun completed!")
    print(f"  Run ID: {result.get('run_id')}")
    print(f"  Answer: {result.get('answer', ['No answer'])[0][:100]}...")
    print(f"  SQL: {result.get('sql_query', 'No SQL')[:100]}...")
    
    # Check step results were stored
    print(f"\nStep results stored:")
    for step_name, results in agent.current_run_step_results.items():
        print(f"  {step_name}: {len(results)} result(s)")
    
    # Clean up
    shutil.rmtree(integration_cache)
else:
    print("Skipping integration test (OPENAI_API_KEY not set)")

Running integration test with per-step best-of-n using gpt-4o-mini...
Checking the model can run locally
OpenAI API is accessible
[Agent] Running agent without visualization
Generated SQL Query:
 SELECT SUM(Total_Sale_Value) as Total_Sales FROM sales WHERE CAST(Sold_Date AS VARCHAR) LIKE '%2021-11%'

Agent response: Total sales in November 2021 amounted to $499,984.43. This figure represents the sum of all sales recorded during that month.
[Agent] Run saved with ID: 2db7675c

Run completed!
  Run ID: 2db7675c
  Answer: Total sales in November 2021 amounted to $499,984.43. This figure represents the sum of all sales re...
  SQL: SELECT SUM(Total_Sale_Value) as Total_Sales FROM sales WHERE CAST(Sold_Date AS VARCHAR) LIKE '%2021-...

Step results stored:
  lookup_sales_data: 1 result(s)
  analyzing_data: 1 result(s)


In [4]:
# Test cache reuse (if OpenAI available)
if OPENAI_AVAILABLE:
    print("Testing cache reuse with gpt-4o-mini...")
    
    cache_dir = tempfile.mkdtemp(prefix="cache_reuse_test_")
    
    config = AgentConfig(model="gpt-4o-mini", provider="openai")
    config.analyzing_data.n = 2
    
    agent = SalesDataAgent(agent_config=config, cache_dir=cache_dir)
    
    # First run - fresh execution
    print("\n--- First run (fresh) ---")
    result1 = agent.run(
        "Show me the sales in Nov 2021",
        no_vis=True,
        save_results=True
    )
    run_id = result1.get('run_id')
    print(f"Run ID: {run_id}")
    
    # Second run - reuse cached lookup, re-run analysis
    print("\n--- Second run (reuse lookup, fresh analysis) ---")
    config.lookup_sales_data.cache_mode = "skip"  # Reuse cached SQL result
    config.analyzing_data.cache_mode = "force_fresh"  # Re-run analysis
    
    result2 = agent.run(
        "Show me the sales in Nov 2021",
        no_vis=True,
        reuse_from=run_id,
        save_results=True
    )
    print(f"Run ID: {result2.get('run_id')}")
    
    # Check cache stats
    print(f"\nCache stats: {agent.cache.get_cache_stats()}")
    
    shutil.rmtree(cache_dir)
else:
    print("Skipping cache reuse test (OPENAI_API_KEY not set)")

Testing cache reuse with gpt-4o-mini...

--- First run (fresh) ---
Checking the model can run locally
OpenAI API is accessible
[Agent] Running agent without visualization
Generated SQL Query:
 SELECT CAST(Sold_Date AS DATE) as Sale_Date, SUM(Qty_Sold) as Total_Qty_Sold, SUM(Total_Sale_Value) as Total_Sale_Value 
FROM sales 
WHERE CAST(Sold_Date AS VARCHAR) LIKE '%2021-11%' 
GROUP BY Sale_Date 
ORDER BY Sale_Date;

Agent response: In November 2021, the total quantity sold was 34,000 units, and the total sales value amounted to $408,000. The sales data covers each day of the month, with the highest single-day sales occurring on November 25, with 5,300 units sold for a total value of $52,004.60.
[Agent] Run saved with ID: a8f04ea5
Run ID: a8f04ea5

--- Second run (reuse lookup, fresh analysis) ---
[Agent] Loaded cached results from run: a8f04ea5
[Agent] Running agent without visualization
Generated SQL Query:
 SELECT CAST(Sold_Date AS DATE) as Sale_Date, SUM(Qty_Sold) as Total_Units_Sold,

In [5]:
# Test step_overrides (if OpenAI available)
if OPENAI_AVAILABLE:
    print("Testing step_overrides with gpt-4o-mini...")
    
    cache_dir = tempfile.mkdtemp(prefix="override_test_")
    
    # Create agent with default config (n=1 for all steps)
    config = AgentConfig(model="gpt-4o-mini", provider="openai")
    agent = SalesDataAgent(agent_config=config, cache_dir=cache_dir)
    
    print(f"Default analyzing_data.n: {agent.agent_config.analyzing_data.n}")
    
    # Run with step overrides for this execution only
    result = agent.run(
        "What is the average sale amount?",
        no_vis=True,
        step_overrides={
            "analyzing_data": {"n": 3, "temp_max": 0.8}
        },
        save_results=True
    )
    
    print(f"\nOverride run completed!")
    print(f"  analyzing_data results stored: {len(agent.current_run_step_results.get('analyzing_data', []))}")
    
    # Verify config was restored
    print(f"  Config after run (should be 1): {agent.agent_config.analyzing_data.n}")
    
    shutil.rmtree(cache_dir)
else:
    print("Skipping step_overrides test (OPENAI_API_KEY not set)")

Testing step_overrides with gpt-4o-mini...
Default analyzing_data.n: 1
Checking the model can run locally
OpenAI API is accessible
[Agent] Running agent without visualization
Generated SQL Query:
 SELECT AVG(Total_Sale_Value) as Average_Sale_Amount FROM sales WHERE Total_Sale_Value IS NOT NULL;

Agent response: The average sale amount is approximately 19.02, as indicated by the data retrieved from the sales records. This value represents the mean of all non-null total sale values in the dataset.
[Agent] Run saved with ID: 3d7775aa

Override run completed!
  analyzing_data results stored: 1
  Config after run (should be 1): 1


## 9. Summary

This notebook tested:

1. **StepConfig** - Per-step configuration with n, temperature range, caching options
2. **AgentConfig** - Container for all step configs with serialization
3. **RunCache** - Result caching with prompt similarity matching
4. **Evaluator factories** - make_csv_evaluator, make_text_evaluator
5. **Agent initialization** - With custom config and cache directory
6. **Backward compatibility** - Old-style initialization still works
7. **Integration** - Full agent run with per-step best-of-n (if Ollama available)
8. **Cache reuse** - Selective re-execution with cached results
9. **Step overrides** - Per-run configuration customization

In [ ]:
print("All tests completed!")